In [3]:
# 1

# Tiandra Taylor

import warnings
warnings.simplefilter("ignore")

In [97]:
# 2

# pre-works
import pandas as pd 
pd.set_option('display.max_colwidth', 90)

import psycopg2

#establish the connection
conn = psycopg2.connect(
    database="loftsvc", 
    user=os.getenv('DB_USER'), 
    password=os.getenv('DB_PASSWORD'), 
    host=os.getenv('DB_HOST'), 
    port= '5432'
)

cursor = conn.cursor()


# fixstamp = null, output exceptid, record, and fixstamp
qry = '''SELECT exceptid, record, fixstamp
        FROM ExceptionLog 
        WHERE fixstamp is NULL;'''

df = pd.read_sql(qry, conn)
print(df)

    exceptid  \
0          1   
1          2   
2          3   
3          5   
4          6   
5          9   
6         11   
7         13   
8         15   
9         16   
10        17   
11        18   
12        19   
13        20   

                                                                                       record  \
0   Elizabeth Marsh||Gettysburg|2024-01-17|721|Faucet dripping, dripping forever, dripping...   
1                Amanda||Hudson|2024-01-17||Mighty cold sucking from thermostat; @#$%*&! fan.   
2                                                Tracey Thompson|doors|Hudson|2024-01-18|893|   
3           Willie Greene||Charleston|2024-01-19|331|Internet running at 270 bits per second.   
4   Teri Perry|windows|Detroit|2024-01-20|486|Stuck and injured @#$%*&!, lock @#$%*&! hand...   
5                              John Clark||Antietam|2024-01-21|105|Unexpected a/c incursions.   
6   Court Boone|connectivity|Charleston|2024-01-22||Slow download wireless for h

In [12]:
# 3

# what's the problem? fix?
print("Rhonda Carter's anomaly: ","\n","- Ensure her name, and all other names in this table, are in the Tenant table.","\n","- Her building value is null.","\t", " -> For all tenants whose building value is null, check to see what range the unit number the resident resides at is and add the corresponding building name in place of the null value.", "\n", "- The date is in the correct format but it is of the year 2024. Date must be within the current calendar year.", "\n", sep="")

print("Willie Greene's anomaly: ","\n", "- The Type value is null.","\t", "-> For all tenants who have a null type, if the description in the record columns indicates the type, update it accordingly, else mark it as other.","\n", "- The date is in the correct format but it is of the year 2024. Date must be within the current calendar year.", "\n", sep="")

print("Jerry Massey's anomaly: ","\n", "- The date is null. This table appears to be in chronological order though. I would add in the date of 2024-01-21, as that is the date for the record above and below.","      ","-> I would follow this same process (if date from above and below record are the same, use that date), for all residents this affects. This does not fix the 'within this calendar year' issue.", "\n", sep="")

print("Gabriela Carter's anomaly: ","\n", "- The type is outlets but that is not allowed. It makes most sense to update this value to 'electrical'.", "\t", " -> For all residents whose type is wrong, I would follow the process described in Willie Green's paragraph.","\n", "- The date is in the correct format but it is of the year 2024. Date must be within the current calendar year.", "\n", sep="")

print("I would not recommend doing this as this will cause data quality issues, but seeing as this assignments purpose is to fix the issues of the data in purgatory to move them to the 'good place', I will be changing the year. If we do not change the year, then we will not resolve all ingestion constraints and won't be able to complete the assignment as expected.", sep="")

Rhonda Carter's anomaly: 
- Ensure her name, and all other names in this table, are in the Tenant table.
- Her building value is null.	 -> For all tenants whose building value is null, check to see what range the unit number the resident resides at is and add the corresponding building name in place of the null value.
- The date is in the correct format but it is of the year 2024. Date must be within the current calendar year.

Willie Greene's anomaly: 
- The Type value is null.	-> For all tenants who have a null type, if the description in the record columns indicates the type, update it accordingly, else mark it as other.
- The date is in the correct format but it is of the year 2024. Date must be within the current calendar year.

Jerry Massey's anomaly: 
- The date is null. This table appears to be in chronological order though. I would add in the date of 2024-01-21, as that is the date for the record above and below.      -> I would follow this same process (if date from above and

In [24]:
# 4

#  add a column to the ServReq table called exceptid

qry = '''ALTER TABLE ServReq ADD COLUMN IF NOT EXISTS exceptid INT'''

cursor.execute(qry)
conn.commit()

# srid and exceptid of the FIRST THREE records in the ServReq table.
qry1 = '''SELECT srid, exceptid
        FROM ServReq 
        LIMIT 3'''

df1 = pd.read_sql(qry1, conn)
print(df1)

   srid exceptid
0     1     None
1     2     None
2     3     None


In [86]:
# 5

# pre-works
from datetime import datetime

# dry run 
issuelst = []
tenantid = 0
prevdate = None
good = True

for i in df.index:
    rc = df['record'].loc[i]
    exceptidcontent = df['exceptid'].loc[i]
    # split
    rc = rc.strip()
    rc = rc.split("|")
    # issueslist be blank
    issueslist = []
    tenantid = 0
    good = True
    # name
    if ' ' not in rc[0]:
        print(f"Can't fix (one name): {rc}") # or no name
        good = False
    # tenant table
    else: 
        #ttble = rc[0].split(" ") # split fname and lname
        qryttbl = """SELECT COUNT(CONCAT(fname, ' ', lname)) as NAME FROM Tenant WHERE CONCAT(fname, ' ', lname) = '%s'""" % (rc[0])
        cursor.execute(qryttbl)
        y = cursor.fetchone()
        if y[0] < 1:
            print(f"Can’t Fix (Unknown Name): {rc}")
            good = False
        else:
            cursor.execute("""SELECT tenantid FROM Tenant WHERE CONCAT(fname, ' ', lname) = '%s'""" % (rc[0]))
            z = cursor.fetchone()
            tenantid = z[0]
    #type
    if len(rc[1]) == 0 or (rc[1]).lower() not in ['connectivity', 'doors', 'electrical', 'hvac', 'plumbing', 'windows', 'other']:
        if 'Internet' in rc[5]:
            qrytype = """SELECT rtid FROM reqtype WHERE typename = 'connectivity'"""
            cursor.execute(qrytype)
            z = cursor.fetchone()
            rc[1] = z[0]
        elif 'outlets' in rc[1]:
            qrytype1 = """SELECT rtid FROM reqtype WHERE typename = 'electrical'"""
            cursor.execute(qrytype1)
            z = cursor.fetchone()
            rc[1] = z[0]
        else:
            print(f"Can’t Fix (Unknown Type): {rc}")
            good = False
    # building value
    if len(rc[2]) == 0 or (rc[2]).lower() not in ['antietam', 'bull run', 'charleston', 'detroit', 'elkins ferry', 'fredericksburg', 'gettysburg', 'hudson']:
        if len(rc[4]) == 0:
            print(f"Can't Fix (No Unit #): {rc}")
            good = False
        elif int(rc[4]) < 101 or int(rc[4]) > 898:
            print(f"Can't Fix (Bad Unit #): {rc}")
            good = False
        else:
            print(rc)
            cursor.execute("""SELECT bldgid, bldgname, unitrange FROM Building;""")
            v = cursor.fetchall()
            for b in v:
                rng = b[2].split("-")
                if int(rc[4]) >= int(rng[0]) and int(rc[4]) <= int(rng[1]):
                    rc[2] = b[0]
                    break   
    # date
    try:
        date_obj = datetime.strptime(rc[3], '%Y-%m-%d')
        prevdate = rc[3]
    except:
        if i == 0: # first record
            print(f"Can't Fix (No Date): {rc}")
            good = False
            continue
        else: 
            rc[3] = prevdate

    # unit num
    if len(rc[4]) == 0:
        print(f"Can't Fix (No Unit #): {rc}")
        good = False
    if len(rc[5]) == 0:
        print(f"Can't Fix (No Description): {rc}")
        good = False
    if good is True:
        qryfinal = """SELECT t.tenantid, r.rtid, b.bldgid, s.reqdate, s.unit, s.reqdescription, s.exceptid 
        FROM Tenant t JOIN ServReq s ON t.tenantid = s.tenantid JOIN Building b on b.bldgid = s.bldgid JOIN ReqType r on r.rtid = s.rtid
        WHERE t.tenantid = '%s' """ % (tenantid)
        cursor.execute(qryfinal)
        v=cursor.fetchall()
        for row in v:
            print(f"tenantid: {row[0]}, rtid: {row[1]}, bldgid: {row[2]}, reqdate: {row[3]}, unit: {row[4]}, reqdescription: {row[5]}, ServReq.exceptid: {row[6]}, ExceptionLog.exceptid: {df['exceptid'].loc[i]}")
            

Can’t Fix (Unknown Type): ['Elizabeth Marsh', '', 'Gettysburg', '2024-01-17', '721', 'Faucet dripping, dripping forever, dripping highly, morosely.']
Can't fix (one name): ['Amanda', '', 'Hudson', '2024-01-17', '', 'Mighty cold sucking from thermostat; @#$%*&! fan.']
Can’t Fix (Unknown Type): ['Amanda', '', 'Hudson', '2024-01-17', '', 'Mighty cold sucking from thermostat; @#$%*&! fan.']
Can't Fix (No Unit #): ['Amanda', '', 'Hudson', '2024-01-17', '', 'Mighty cold sucking from thermostat; @#$%*&! fan.']
Can't Fix (No Description): ['Tracey Thompson', 'doors', 'Hudson', '2024-01-18', '893', '']
['Rhonda Carter', 'hvac', '', '2024-01-18', '268', 'Aircon sounds like a ghost lives inside it.']
tenantid: 455, rtid: 3, bldgid: 5, reqdate: 2024-01-17, unit: 522, reqdescription: Slumlord bulb bulb whine whine a., ServReq.exceptid: None, ExceptionLog.exceptid: 4
Can’t Fix (Unknown Name): ['Teri Perry', 'windows', 'Detroit', '2024-01-20', '486', 'Stuck and injured @#$%*&!, lock @#$%*&! handle wi

In [94]:
# 6

# pre-works
from datetime import datetime

# dry run 
issuelst = []
tenantid = 0
prevdate = None
good = True

for i in df.index:
    rc = df['record'].loc[i]
    exceptidcontent = df['exceptid'].loc[i]
    # split
    rc = rc.strip()
    rc = rc.split("|")
    # issueslist be blank
    issueslist = []
    tenantid = 0
    good = True
    # name
    if ' ' not in rc[0]:
        print(f"Can't fix (one name): {rc}") # or no name
        good = False
    # tenant table
    else: 
        #ttble = rc[0].split(" ") # split fname and lname
        qryttbl = """SELECT COUNT(CONCAT(fname, ' ', lname)) as NAME FROM Tenant WHERE CONCAT(fname, ' ', lname) = '%s'""" % (rc[0])
        cursor.execute(qryttbl)
        y = cursor.fetchone()
        if y[0] < 1:
            print(f"Can’t Fix (Unknown Name): {rc}")
            good = False
        else:
            cursor.execute("""SELECT tenantid FROM Tenant WHERE CONCAT(fname, ' ', lname) = '%s'""" % (rc[0]))
            z = cursor.fetchone()
            tenantid = z[0]
    #type
    if len(rc[1]) == 0 or (rc[1]).lower() not in ['connectivity', 'doors', 'electrical', 'hvac', 'plumbing', 'windows', 'other']:
        if 'Internet' in rc[5]:
            qrytype = """SELECT rtid FROM reqtype WHERE typename = 'connectivity'"""
            cursor.execute(qrytype)
            z = cursor.fetchone()
            rc[1] = z[0]
        elif 'outlets' in rc[1]:
            qrytype1 = """SELECT rtid FROM reqtype WHERE typename = 'electrical'"""
            cursor.execute(qrytype1)
            z = cursor.fetchone()
            rc[1] = z[0]
        else:
            print(f"Can’t Fix (Unknown Type): {rc}")
            good = False
    # building value
    if len(rc[2]) == 0 or (rc[2]).lower() not in ['antietam', 'bull run', 'charleston', 'detroit', 'elkins ferry', 'fredericksburg', 'gettysburg', 'hudson']:
        if len(rc[4]) == 0:
            print(f"Can't Fix (No Unit #): {rc}")
            good = False
        elif int(rc[4]) < 101 or int(rc[4]) > 898:
            print(f"Can't Fix (Bad Unit #): {rc}")
            good = False
        else:
            cursor.execute("""SELECT bldgid, bldgname, unitrange FROM Building;""")
            v = cursor.fetchall()
            for b in v:
                rng = b[2].split("-")
                if int(rc[4]) >= int(rng[0]) and int(rc[4]) <= int(rng[1]):
                    rc[2] = b[0]
                    break   
    # date
    try:
        date_obj = datetime.strptime(rc[3], '%Y-%m-%d')
        prevdate = rc[3]
    except:
        if i == 0: # first record
            print(f"Can't Fix (No Date): {rc}")
            good = False
            continue
        else: 
            rc[3] = prevdate

    # unit num
    if len(rc[4]) == 0:
        print(f"Can't Fix (No Unit #): {rc}")
        good = False
    if len(rc[5]) == 0:
        print(f"Can't Fix (No Description): {rc}")
        good = False
    if good is True:
        # Look up rtid from ReqType table using the type name
        try:
            qry_rtid = """SELECT rtid FROM ReqType WHERE typename = '%s'""" % (rc[1].lower())
            cursor.execute(qry_rtid)
            rtid_result = cursor.fetchone()
            if rtid_result:
                rc[1] = rtid_result[0]
        except:
            continue
            
        # Look up bldgid from Building table using the building name
        try:
            qry_bldgid = """SELECT bldgid FROM Building WHERE bldgname = '%s'""" % (rc[2])
            cursor.execute(qry_bldgid)
            bldgid_result = cursor.fetchone()
            if bldgid_result:
                rc[2] = bldgid_result[0]
        except:
            continue
        
        # INSERT the fixed record
        insert_query = """INSERT INTO ServReq (tenantid, rtid, bldgid, reqdate, unit, reqdescription, exceptid) 
                          VALUES ('%s', '%s', '%s', '%s', '%s', '%s', '%s')""" % (tenantid, rc[1], rc[2], rc[3], rc[4], rc[5], exceptidcontent)
        cursor.execute(insert_query)
        conn.commit()
        
        # Output the record that was added
        print(f"Record Added: {tenantid}, {rc[1]}, {rc[2]}, {rc[3]}, {rc[4]}, {rc[5]}, {exceptidcontent}")
        
        # Update the ExceptionLog table
        update_query = """UPDATE ExceptionLog SET fixstamp = NOW() WHERE exceptid = '%s'""" % (exceptidcontent)
        cursor.execute(update_query)
        conn.commit()

Can’t Fix (Unknown Type): ['Elizabeth Marsh', '', 'Gettysburg', '2024-01-17', '721', 'Faucet dripping, dripping forever, dripping highly, morosely.']
Can't fix (one name): ['Amanda', '', 'Hudson', '2024-01-17', '', 'Mighty cold sucking from thermostat; @#$%*&! fan.']
Can’t Fix (Unknown Type): ['Amanda', '', 'Hudson', '2024-01-17', '', 'Mighty cold sucking from thermostat; @#$%*&! fan.']
Can't Fix (No Unit #): ['Amanda', '', 'Hudson', '2024-01-17', '', 'Mighty cold sucking from thermostat; @#$%*&! fan.']
Can't Fix (No Description): ['Tracey Thompson', 'doors', 'Hudson', '2024-01-18', '893', '']
Record Added: 455, 5, 2, 2024-01-18, 268, Aircon sounds like a ghost lives inside it., 4
Can’t Fix (Unknown Name): ['Teri Perry', 'windows', 'Detroit', '2024-01-20', '486', 'Stuck and injured @#$%*&!, lock @#$%*&! handle will not open.']
Record Added: 504, 2, 3, 2024-01-21, 345, Door bashed with an axe bashed with an axe trapped locked inside., 7
Record Added: 255, 6, 4, 2024-01-21, 431, No hot w

In [98]:
# 7

qrychk = ''' SELECT srid, tenantid, rtid, bldgid, reqdate, unit, exceptid FROM ServReq WHERE exceptid IS NOT NULL;'''
dfcheck = pd.read_sql(qrychk, conn)
print(dfcheck)

   srid  tenantid  rtid  bldgid     reqdate  unit  exceptid
0   116       455     5       2  2024-01-18   268         4
1   117       504     2       3  2024-01-21   345         7
2   118       255     6       4  2024-01-21   431         8
3   119       538     1       7  2024-01-22   753        10
4   120       116     2       1  2024-01-23   176        12
5   121       495     3       4  2024-01-25   413        14


In [99]:
# 8

qrychk2 = '''SELECT exceptid, record, fixstamp FROM ExceptionLog WHERE fixstamp IS NOT NULL;'''

dfcheck2 = pd.read_sql(qrychk2, conn)
print(dfcheck2)

   exceptid  \
0         4   
1         7   
2         8   
3        10   
4        12   
5        14   

                                                                                      record  \
0             Rhonda Carter|hvac||2024-01-18|268|Aircon sounds like a ghost lives inside it.   
1  Victoria Ward|doors||2024-01-21|345|Door bashed with an axe bashed with an axe trapped...   
2                  Jerry Massey|plumbing|Detroit||431|No hot water. 11 days. Hope dwindling.   
3  Shannon Ramos|connectivity||2024-01-22|753|Slow download of wireless, hopelessness of ...   
4  Darryl Vazquez|doors|Antietam|2024-01-23|176|Door wont shut, door refuses the shutting...   
5             Laurie Lopez|electrical||2024-01-25|413|Hopeless light switches demean us all.   

                    fixstamp  
0 2025-11-01 03:52:44.949735  
1 2025-11-01 03:52:45.455922  
2 2025-11-01 03:52:45.778431  
3 2025-11-01 03:52:46.200401  
4 2025-11-01 03:52:46.536800  
5 2025-11-01 03:52:46.939762  
